# 02 — Build the FAISS Job Index & Semantic Search

1. Load the job CSV (`config.JOBS_CSV`) with pandas. Combine `jobtitle + skills + jobdescription` into one text per job.
2. Embed those job texts and build a **FAISS** index. Save it to `config.JOBS_INDEX_DIR` so the app loads it instead of rebuilding.
3. Embed a candidate profile and run a **top-N** similarity search — return the closest jobs with scores.
4. Sanity-check: do the top jobs actually match the profile? Move the load/query code into `src/search/job_search.py`.

Start with a few thousand jobs while developing so embedding is fast and cheap.

In [1]:
# ============================================================
# SMART HIRE - NOTEBOOK 02
# FAISS JOB SEARCH
#
# 22,000 Jobs
#       ↓
# 22,000 Texts
#       ↓
# Local Embeddings (NOT Gemini)
#       ↓
# FAISS Index
#       ↓
# Candidate Profile
#       ↓
# Top-N Similar Jobs
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer


# ============================================================
# 2. FIND PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project folder:", PROJECT_ROOT)


# ============================================================
# 3. IMPORT PROJECT CONFIG
# ============================================================

from src import config


# ============================================================
# 4. LOAD ALL JOBS
# ============================================================

jobs_file = config.JOBS_CSV

jobs_df = pd.read_csv(jobs_file)

print("\nTotal jobs:", len(jobs_df))


# ============================================================
# 5. CREATE ONE TEXT PER JOB
#
# According to the project instruction:
# Combine jobtitle + skills + jobdescription
# into ONE text per job.
# ============================================================

job_texts = []

for _, row in jobs_df.iterrows():

    job_title = (
        str(row["jobtitle"])
        if pd.notna(row["jobtitle"])
        else ""
    )

    skills = (
        str(row["skills"])
        if pd.notna(row["skills"])
        else ""
    )

    job_description = (
        str(row["jobdescription"])
        if pd.notna(row["jobdescription"])
        else ""
    )

    # ONE combined text for ONE job
    job_text = (
        job_title + " " +
        skills + " " +
        job_description
    )

    job_texts.append(job_text)


# ============================================================
# 6. VERIFY 22,000 TEXTS
# ============================================================

print("Total jobs:", len(jobs_df))
print("Total job texts:", len(job_texts))

print("\nFirst job text:")
print(job_texts[0])


# ============================================================
# 7. LOAD LOCAL EMBEDDING MODEL
#
# This DOES NOT use Gemini API.
# ============================================================

print("\nLoading local embedding model...")

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Local embedding model loaded successfully.")


# ============================================================
# 8. CREATE EMBEDDINGS FOR ALL 22,000 JOBS
# ============================================================

print("\nCreating embeddings for all jobs...")

job_embeddings = model.encode(
    job_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

job_embeddings = np.asarray(
    job_embeddings,
    dtype=np.float32
)


# ============================================================
# 9. CHECK EMBEDDINGS
# ============================================================

print("\n" + "=" * 60)
print("JOB EMBEDDINGS CREATED")
print("=" * 60)

print("Number of jobs:", len(jobs_df))
print("Number of job texts:", len(job_texts))
print("Number of embeddings:", len(job_embeddings))
print("Embedding shape:", job_embeddings.shape)
print("Embedding dimension:", job_embeddings.shape[1])


# ============================================================
# 10. CREATE FAISS INDEX
# ============================================================

print("\nCreating FAISS index...")

embedding_dimension = job_embeddings.shape[1]

# Because embeddings are normalized,
# Inner Product works as cosine similarity.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(job_embeddings)

print("FAISS index created.")
print("Vectors stored in FAISS:", index.ntotal)


# ============================================================
# 11. CREATE VECTORSTORE FOLDER
# ============================================================

config.JOBS_INDEX_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 12. SAVE FAISS INDEX
# ============================================================

index_file = (
    config.JOBS_INDEX_DIR /
    "index.faiss"
)

faiss.write_index(
    index,
    str(index_file)
)

print("\nFAISS index saved to:")
print(index_file)


# ============================================================
# 13. SAVE JOB DATA
#
# This keeps the original job information so that
# FAISS result number can be matched to the actual job.
# ============================================================

jobs_metadata_file = (
    config.JOBS_INDEX_DIR /
    "jobs.csv"
)

jobs_df.to_csv(
    jobs_metadata_file,
    index=False
)

print("\nJob information saved to:")
print(jobs_metadata_file)


# ============================================================
# 14. CREATE CANDIDATE PROFILE
# ============================================================

candidate_profile = """
Skills: Python, Machine Learning, Data Science, SQL,
Pandas, NumPy, Scikit-learn, TensorFlow.

Experience: Experience in data analysis, machine learning,
and building predictive models.

Education: Computer Science / Information Technology.

Target Role: Machine Learning Engineer / AI Engineer.
"""


# ============================================================
# 15. EMBED CANDIDATE PROFILE
#
# IMPORTANT:
# Use the SAME local model used for the jobs.
# ============================================================

print("\nCreating candidate embedding...")

candidate_embedding = model.encode(
    [candidate_profile],
    normalize_embeddings=True
)

candidate_embedding = np.asarray(
    candidate_embedding,
    dtype=np.float32
)

print(
    "Candidate embedding shape:",
    candidate_embedding.shape
)


# ============================================================
# 16. TOP-N SEMANTIC SEARCH
# ============================================================

TOP_N = config.TOP_N_JOBS

print(
    f"\nSearching for top {TOP_N} matching jobs..."
)

scores, indices = index.search(
    candidate_embedding,
    TOP_N
)


# ============================================================
# 17. DISPLAY TOP-N JOBS
# ============================================================

print("\n" + "=" * 70)
print("TOP MATCHING JOBS")
print("=" * 70)

results = []

for rank, (score, idx) in enumerate(
    zip(scores[0], indices[0]),
    start=1
):

    job = jobs_df.iloc[idx]

    result = {
        "rank": rank,
        "similarity_score": float(score),
        "jobtitle": job["jobtitle"],
        "skills": job["skills"],
        "jobdescription": job["jobdescription"]
    }

    results.append(result)

    print("\n" + "-" * 70)

    print("Rank:", rank)

    print(
        "Similarity Score:",
        f"{score:.4f}"
    )

    print(
        "Job Title:",
        job["jobtitle"]
    )

    print(
        "Skills:",
        job["skills"]
    )


# ============================================================
# 18. CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("SEMANTIC JOB SEARCH COMPLETED")
print("=" * 70)

print(
    results_df[
        [
            "rank",
            "similarity_score",
            "jobtitle"
        ]
    ]
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)

print("Total jobs processed:", len(jobs_df))
print("Total texts created:", len(job_texts))
print("Total embeddings:", len(job_embeddings))
print("FAISS vectors:", index.ntotal)
print(
    "Embedding dimension:",
    job_embeddings.shape[1]
)

print("\nFAISS index:")
print(index_file)

print("\nJob metadata:")
print(jobs_metadata_file)

c:\Users\koppa\SmartHire-GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project folder: c:\Users\koppa\SmartHire-GenAI

Total jobs: 22000
Total jobs: 22000
Total job texts: 22000

First job text:
Walkin Data Entry Operator (night Shift) ITES Job Description   Send me Jobs like this Qualifications: - == > 10th To Graduation & Any Skill: - == > Basic Computer Knowledge Job Requirement : - == > System or Laptop Type of job: - == > Full Time or Part time Languages : - == > Tamil & English. Experience : - == > Freshers & Experience payment details: - 1 form per day 5/- 10 form per day 50/- 100 form per day 500/- monthly you can earn 15000/- per month Selection Process: - == > Easy Selection Process,So What Are You Waiting For? Apply Now & Grab Best Opportunity To Make Your Carrier & To Improve Your Earing Skills. More detail contact Mr Hari 8678902528 9003010282 Salary:INR 1,50,000 - 2,25,000 P.A Industry: Media / Entertainment / Internet Functional Area: ITES , BPO , KPO , LPO , Customer Service , Operations Role Category:Other Role:Fresher Keyskills English T

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5401.52it/s]


Local embedding model loaded successfully.

Creating embeddings for all jobs...


Batches: 100%|██████████| 688/688 [12:58<00:00,  1.13s/it]



JOB EMBEDDINGS CREATED
Number of jobs: 22000
Number of job texts: 22000
Number of embeddings: 22000
Embedding shape: (22000, 384)
Embedding dimension: 384

Creating FAISS index...
FAISS index created.
Vectors stored in FAISS: 22000

FAISS index saved to:
C:\Users\koppa\SmartHire-GenAI\vectorstore\jobs_faiss\index.faiss

Job information saved to:
C:\Users\koppa\SmartHire-GenAI\vectorstore\jobs_faiss\jobs.csv

Creating candidate embedding...
Candidate embedding shape: (1, 384)

Searching for top 5 matching jobs...

TOP MATCHING JOBS

----------------------------------------------------------------------
Rank: 1
Similarity Score: 0.7154
Job Title: Data Scientist Machine Learning
Skills: Analytics & Business Intelligence

----------------------------------------------------------------------
Rank: 2
Similarity Score: 0.6853
Job Title: Data Scientist - Machine Learning/nlp
Skills: IT Software - Embedded

----------------------------------------------------------------------
Rank: 3
Similar